# Tests Unitaires en Python : `unittest` et `pytest`

Les tests unitaires permettent de vérifier automatiquement que **chaque fonction** se comporte comme prévu.

# 1. Pourquoi écrire des tests unitaires ?

Pour devenir un développeur professionnel, qui push en Prod !!!

Un **test unitaire** vérifie le comportement d’une petite portion du code (souvent une fonction) :

1. on fixe une **entrée**,  
2. on définit un **résultat attendu**,  
3. on laisse le test vérifier que la fonction retourne ce résultat.

### Pourquoi c'est important en data science ?
- vos tâches impliquent souvent des transformations, filtrages, normalisations, métriques…  
- un test garantit qu’une modification de code n’introduit pas de bug,  
- vous pourriez colaborer avec d'autres developpeurs, et lorsque chacun modifie des fonctions, les problemes commencent a arriver... Sauf si ces fonctions sont chaque fois testées en suivant un process donné

Au final... Dès qu’une fonction contient de la logique (conditions, exceptions, calculs…), elle mérite un test.
- Inutile de tester toutes les situations possibles
- Il faut se concentrer sur les tests les plus significatifs

# Premier Exemple

Dans un projet, on définit une classe `PriceCalculator` avec une méthode qui calcule le prix final d'un produit suite a une remise.

In [11]:
class PriceCalculator:
    """Calcule le prix final après application d'une remise."""
    
    def calculate_final_price(self, price, discount_percent):        
        discount_amount = price * (discount_percent / 100)
        return price - discount_amount

In [2]:
mon_calculator = PriceCalculator()

In [3]:
mon_calculator.calculate_final_price(price=100, discount_percent=15)

85.0

On vourait a présent tester si cette classe fonctionne correctement

# Tests unitaires avec le module standard `unittest`

`unittest` est inclus dans la bibliothèque standard de Python.  
Idée principale :

1. Créer une **classe de test** qui hérite de `unittest.TestCase`.
2. Écrire des méthodes dont le nom commence par `test_`.
3. Utiliser les **méthodes d’assertion** (`assertEqual`, `assertRaises`, etc.) pour vérifier les résultats.

Schéma minimal :

```python
import unittest

class TestQuelqueChose(unittest.TestCase):
    def test_un_cas(self):
        self.assertEqual(fonction(...), valeur_attendue)
```

### Premier test unitaire avec unittest

In [4]:
import unittest

In [5]:
class TestPriceCalculator(unittest.TestCase):
    
    def test_no_discount(self):
        """Test : pas de remise, prix reste identique."""
        result = PriceCalculator().calculate_final_price(100, 0)
        self.assertEqual(result, 100)

In [6]:
tests = unittest.TestLoader().loadTestsFromTestCase(TestPriceCalculator)

unittest.TextTestRunner(verbosity=2).run(tests)

test_no_discount (__main__.TestPriceCalculator.test_no_discount)
Test : pas de remise, prix reste identique. ... ok

----------------------------------------------------------------------
Ran 1 test in 0.002s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

### Ajout d'autres tests

In [ ]:
class TestPriceCalculator(unittest.TestCase):
    """Tests pour le calculateur de prix."""
    
    def test_no_discount(self):
        """Test : pas de remise, prix reste identique."""
        result = PriceCalculator().calculate_final_price(100, 0)
        self.assertEqual(result, 100)
    
    def test_fifty_percent_discount(self):
        """Test : 50% de remise."""
        result = PriceCalculator().calculate_final_price(100, 50)
        self.assertEqual(result, 50)

    def test_full_discount(self):
        """Test : 100% de remise."""
        result = PriceCalculator().calculate_final_price(100, 100)
        self.assertEqual(result, 0)

In [10]:
tests = unittest.TestLoader().loadTestsFromTestCase(TestPriceCalculator)
unittest.TextTestRunner(verbosity=2).run(tests)

test_fifty_percent_discount (__main__.TestPriceCalculator.test_fifty_percent_discount)
Test : 50% de remise. ... FAIL
test_full_discount (__main__.TestPriceCalculator.test_full_discount)
Test : 100% de remise. ... FAIL
test_no_discount (__main__.TestPriceCalculator.test_no_discount)
Test : pas de remise, prix reste identique. ... FAIL

FAIL: test_fifty_percent_discount (__main__.TestPriceCalculator.test_fifty_percent_discount)
Test : 50% de remise.
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/tmp/ipykernel_157890/817617032.py", line 12, in test_fifty_percent_discount
    self.assertEqual(result, 50)
AssertionError: 49.0 != 50

FAIL: test_full_discount (__main__.TestPriceCalculator.test_full_discount)
Test : 100% de remise.
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/tmp/ipykernel_157890/817617032.py", line 17, in test_full_discount
    self.assert

<unittest.runner.TextTestResult run=3 errors=0 failures=3>

Pour ne pas devoir créer une instance de notre classe chaque fois que l'on effectue un test, on utiliser la méthode `setUp(self)` afin de définir notre instance au tout début de la séquence de tests

In [ ]:
class TestPriceCalculator(unittest.TestCase):
    """Tests pour le calculateur de prix."""
    
    def setUp(self):
        """Préparation avant chaque test - créer une instance."""
        self.calculator = PriceCalculator()
    
    def test_no_discount(self):
        """Test : pas de remise, prix reste identique."""
        result = self.calculator.calculate_final_price(100, 0)
        self.assertEqual(result, 100)
    
    def test_fifty_percent_discount(self):
        """Test : 50% de remise."""
        result = self.calculator.calculate_final_price(100, 50)
        self.assertEqual(result, 50)
    
    def test_full_discount(self):
        """Test : 100% de remise."""
        result = self.calculator.calculate_final_price(100, 100)
        self.assertEqual(result, 0)


In [ ]:
suite = unittest.TestLoader().loadTestsFromTestCase(TestPriceCalculator)
unittest.TextTestRunner(verbosity=2).run(suite)

## Mais pourquoi ne pas utiliser un Print ?

L'objectif ici n'est pas de "Voir" le résultat de nos fonctions...

... Mais de savoir de maniere **immédiate** si celles-ci ont le comportement voulu.

- Cela accelere considerablement le temps de développement
- Cela permet de repérer immédiatement s'il y a des bugs dans nos programmes
- Et cela facilite grandement notre travail quand on modifie ainsi des programmes complexes, faisant intervenir des centaines de fonctions

# Méthodes d’assertion utiles dans `unittest`

Quelques assertions fréquentes :

| Méthode                          | Rôle                                                          |
|----------------------------------|---------------------------------------------------------------|
| `assertEqual(a, b)`              | `a == b`                                                      |
| `assertNotEqual(a, b)`           | `a != b`                                                      |
| `assertTrue(x)` / `assertFalse(x)` | booléen attendu                                              |
| `assertIn(a, b)` / `assertNotIn(a, b)` | appartenance à une collection                            |
| `assertIsNone(x)` / `assertIsNotNone(x)` | `x is None` ou non                                    |
| `assertRaises(Exception, f, ...)` | vérifie qu’un appel lève bien une exception                  |

Lisibilité : l’assertion doit exprimer **clairement l’intention du test**.

## Synergie avec les Exceptions : assertRaises

Dans notre exemple, on pourrait vouloir intégrer a notre fonction des Exceptions pour la protéger de certaines situations.

Exemples : 

- Un prix négatif
- Une remise supérieure a 100%
- Une remise inférieure a 0%

In [12]:
class PriceCalculator:
    """Calcule le prix final après application d'une remise."""
    
    def calculate_final_price(self, price, discount_percent):
        """
        Calcule le prix final après remise.
        
        Args:
            price: Prix initial (doit être positif)
            discount_percent: Pourcentage de remise (0-100)
        
        Returns:
            Prix final après remise
        """
        if price < 0:
            raise ValueError("Le prix ne peut pas être négatif")
        if discount_percent < 0 or discount_percent > 100:
            raise ValueError("La remise doit être entre 0 et 100")
        
        discount_amount = price * (discount_percent / 100)
        return price - discount_amount

In [13]:
mon_calculator = PriceCalculator()

mon_calculator.calculate_final_price(100, 120)

ValueError: La remise doit être entre 0 et 100

Ainsi, on peut également vouloir créer un test pour vérifier que notre fonction souleve bel et bien une exception quand elle rencontre certains cas de figure

In [14]:
class TestPriceCalculator(unittest.TestCase):
    """Tests pour le calculateur de prix."""
    
    def setUp(self):
        """Préparation avant chaque test - créer une instance."""
        self.calculator = PriceCalculator()
    
    # def test_no_discount(self):
    #     """Test : pas de remise, prix reste identique."""
    #     result = self.calculator.calculate_final_price(100, 0)
    #     self.assertEqual(result, 100)
    
    # def test_fifty_percent_discount(self):
    #     """Test : 50% de remise."""
    #     result = self.calculator.calculate_final_price(100, 50)
    #     self.assertEqual(result, 50)
    
    # def test_full_discount(self):
    #     """Test : 100% de remise."""
    #     result = self.calculator.calculate_final_price(100, 100)
    #     self.assertEqual(result, 0)
    
    def test_negative_price_raises_error(self):
        """Test : prix négatif doit lever une exception."""
        with self.assertRaises(ValueError):
            self.calculator.calculate_final_price(-10, 20)
    
    def test_invalid_discount_raises_error(self):
        """Test : remise invalide doit lever une exception."""
        with self.assertRaises(ValueError):
            self.calculator.calculate_final_price(100, 150)

In [15]:
suite = unittest.TestLoader().loadTestsFromTestCase(TestPriceCalculator)
unittest.TextTestRunner(verbosity=2).run(suite)

test_invalid_discount_raises_error (__main__.TestPriceCalculator.test_invalid_discount_raises_error)
Test : remise invalide doit lever une exception. ... ok
test_negative_price_raises_error (__main__.TestPriceCalculator.test_negative_price_raises_error)
Test : prix négatif doit lever une exception. ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.004s

OK


<unittest.runner.TextTestResult run=2 errors=0 failures=0>

# 5. Exécuter les tests `unittest` dans un vrai projet

En pratique, on organise souvent les fichiers comme ceci :

```text
project/
    src/
        mon_code.py
    tests/
        test_mon_code.py
```

Exemple minimal de fichier test_mon_code.py :

```python
import unittest
from src.mon_code import ma_function

class TestAppliquerLog(unittest.TestCase):
    def test_log_classique(self):
        self.assertEqual(ma_function(100, 0.2), 80.0)

if __name__ == "__main__":
    unittest.main()
```

Puis on lance :

```bash
python -m unittest tests/test_mon_code.py
```

# 6. Introduction à `pytest`

`pytest` est un framework de tests externe qui se veut plus concis, lisible et extensible que `unittest`.

Principes clés de base :

- les tests sont de simples **fonctions** qui commencent par `test_`,
- on utilise l’**assert Python natif** (`assert x == y`),
- aucune classe n’est nécessaire pour les cas simples.

### Exemple équivalent avec pytest 

Dans un fichier par exemple `test_log_pytest.py`

In [ ]:
def test_log_classique_pytest():
    resultat = safe_log(10)
    import math
    assert resultat == math.log(10)

def test_log_unitaire_pytest():
    # log(1) = 0
    assert safe_log(1) == 0.0

def test_valeur_negative_pytest():
    import pytest
    with pytest.raises(ValueError):
        safe_log(-10)

# 7. Lancer `pytest`

Avec un fichier `test_log_pytest.py` :

```bash
pytest
# ou pour plus de détails
pytest -v
```

Par défaut, pytest :
- cherche les fichiers dont le nom commence par `test_`,
- exécute toutes les fonctions `test_*` à l’intérieur,
- affiche un résumé clair : tests passés, échoués, erreurs.


## 9. Organisation d’un projet de tests avec pytest

Structure typique :

```text
project/
    src/
        prix.py
    tests/
        test_prix.py
        test_autre_module.py
```

Commandes usuelles :

```bash
pytest           # lance tous les tests
pytest tests/test_prix.py -v   # lance les tests d’un fichier spécifique
pytest -k "log"  # ne lance que les tests dont le nom contient "log"
```

Quelques avantages de pytest :
- syntaxe plus légère,
- paramétrisation puissante,
- système de fixtures très flexible,
- écosystème riche de plugins (couverture, tests parallèles, etc.).

# Au final

- `unittest` : framework standard, basé sur des **classes de test** et des **méthodes d’assertion**.
- `pytest` : framework externe, basé sur des **fonctions de test** et le mot-clé `assert`.
- Dans les deux cas :
  - on isole une **unité de code**,
  - on vérifie automatiquement des **cas nominaux** et des **cas d’erreur**,
  - on gagne en **fiabilité** lors de l’évolution du code.

Pour progresser : ajoutez des tests à vos projets existants, en commençant par les fonctions les plus critiques.

# Exercices

## Exercice 1

Nous avons créer une classe `EmailValidator` qui permet, comme son nom l'indique, de valider un email :)

In [ ]:
import re

class EmailValidator:
    """Valide les adresses email."""
    
    def is_valid(self, email):
        """Vérifie si l'email est valide."""
        if not email or not isinstance(email, str):
            return False
        
        pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
        return bool(re.match(pattern, email))
    
    def extract_domain(self, email):
        """Extrait le domaine de l'email."""
        return email.split('@')[1]

## Consignes

Créez une classe TestEmailValidator qui teste les scénarios suivants :
Tests pour is_valid() :

- Tester qu'un email valide standard (ex: "user@example.com") retourne True
- Tester qu'un email avec des caractères spéciaux autorisés (ex: "user.name+tag@example.co.uk") retourne True
- Tester qu'un email sans @ retourne False
- Tester qu'un email sans domaine (ex: "user@") retourne False
- Tester qu'un email sans extension (ex: "user@example") retourne False
- Tester qu'une chaîne vide retourne False
- Tester que None retourne False

Tests pour extract_domain() :
- Tester l'extraction correcte du domaine pour un email valide
- Tester qu'une exception ValueError est levée pour un email invalide

**Indices**: Assertions à utiliser : `assertTrue()`, `assertFalse()`, `assertEqual()`, `assertRaises()`

## Correction

In [ ]:
import re

class EmailValidator:
    """Valide les adresses email."""
    
    def is_valid(self, email):
        """Vérifie si l'email est valide."""
        if not email or not isinstance(email, str):
            return False
        
        pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
        return bool(re.match(pattern, email))
    
    def extract_domain(self, email):
        """Extrait le domaine de l'email."""
        if not self.is_valid(email):
            raise ValueError("Email invalide")
        return email.split('@')[1]

In [ ]:
class TestEmailValidator(unittest.TestCase):
    """Tests pour le validateur d'email."""
    
    def setUp(self):
        self.validator = EmailValidator()
    
    def test_valid_email(self):
        """Test : email valide."""
        self.assertTrue(self.validator.is_valid("user@example.com"))
    
    def test_invalid_email_no_at(self):
        """Test : email sans @."""
        self.assertFalse(self.validator.is_valid("userexample.com"))
    
    def test_invalid_email_no_domain(self):
        """Test : email sans domaine."""
        self.assertFalse(self.validator.is_valid("user@"))
    
    def test_empty_email(self):
        """Test : email vide."""
        self.assertFalse(self.validator.is_valid(""))
    
    def test_extract_domain_valid(self):
        """Test : extraction du domaine."""
        domain = self.validator.extract_domain("user@example.com")
        self.assertEqual(domain, "example.com")
    
    def test_extract_domain_invalid_email(self):
        """Test : extraction sur email invalide doit échouer."""
        with self.assertRaises(ValueError):
            self.validator.extract_domain("invalid-email")

## Exercice 2:

Ici, une Classe `ShoppingCart` permet de créer des paniers d'achats, d'ajouter des élements, de calculer la valeur total, et de vider le panier...

In [ ]:
class ShoppingCart:
    """Panier d'achats."""
    
    def __init__(self):
        self.items = []
    
    def add_item(self, name, price, quantity=1):
        """Ajoute un article au panier."""        
        self.items.append({
            'name': name,
            'price': price,
            'quantity': quantity
        })
    
    def get_total(self):
        """Calcule le total du panier."""
        return sum(item['price'] * item['quantity'] for item in self.items)
    
    def get_item_count(self):
        """Retourne le nombre total d'articles."""
        return sum(item['quantity'] for item in self.items)
    
    def clear(self):
        """Vide le panier."""
        self.items = []

## Consignes

Créez une classe TestShoppingCart qui teste les scénarios suivants :
Tests de l'état initial :

- Tester qu'un nouveau panier a un total de 0
- Tester qu'un nouveau panier a un compteur d'articles à 0

Tests d'ajout d'articles :
- Tester l'ajout d'un seul article (vérifier total et compteur)
- Tester l'ajout de plusieurs articles différents
- Tester l'ajout d'un article avec une quantité > 1
- Tester qu'ajouter un article avec un prix négatif lève une ValueError
- Tester qu'ajouter un article avec une quantité de 0 lève une ValueError
- Tester qu'ajouter un article avec une quantité négative lève une ValueError

Tests de manipulation du panier :
- Tester que clear() vide complètement le panier
- Tester la suppression d'un article spécifique avec remove_item()
- Tester que supprimer un article inexistant n'affecte pas le panier


**Indices**: Assertions à utiliser : `assertEqual()`, `assertAlmostEqual()`, `assertRaises()`, `assertGreater()`

Conseil : Utilisez setUp() pour créer un nouveau panier avant chaque test !

## Correction

In [ ]:
class ShoppingCart:
    """Panier d'achats."""
    
    def __init__(self):
        self.items = []
    
    def add_item(self, name, price, quantity=1):
        """Ajoute un article au panier."""
        if price < 0:
            raise ValueError("Le prix ne peut pas être négatif")
        if quantity <= 0:
            raise ValueError("La quantité doit être positive")
        
        self.items.append({
            'name': name,
            'price': price,
            'quantity': quantity
        })
    
    def get_total(self):
        """Calcule le total du panier."""
        return sum(item['price'] * item['quantity'] for item in self.items)
    
    def get_item_count(self):
        """Retourne le nombre total d'articles."""
        return sum(item['quantity'] for item in self.items)
    
    def clear(self):
        """Vide le panier."""
        self.items = []

In [ ]:
class TestShoppingCart(unittest.TestCase):
    """Tests pour le panier d'achats."""
    
    def setUp(self):
        """Créer un nouveau panier avant chaque test."""
        self.cart = ShoppingCart()
    
    def test_new_cart_is_empty(self):
        """Test : nouveau panier est vide."""
        self.assertEqual(self.cart.get_total(), 0)
        self.assertEqual(self.cart.get_item_count(), 0)
    
    def test_add_single_item(self):
        """Test : ajouter un article."""
        self.cart.add_item("Livre", 15.99)
        self.assertEqual(self.cart.get_total(), 15.99)
        self.assertEqual(self.cart.get_item_count(), 1)
    
    def test_add_multiple_items(self):
        """Test : ajouter plusieurs articles."""
        self.cart.add_item("Livre", 15.99)
        self.cart.add_item("Stylo", 2.50, quantity=3)
        
        expected_total = 15.99 + (2.50 * 3)
        self.assertAlmostEqual(self.cart.get_total(), expected_total, places=2)
        self.assertEqual(self.cart.get_item_count(), 4)
    
    def test_clear_cart(self):
        """Test : vider le panier."""
        self.cart.add_item("Livre", 15.99)
        self.cart.clear()
        self.assertEqual(self.cart.get_total(), 0)
        self.assertEqual(len(self.cart.items), 0)
    
    def test_negative_price_raises_error(self):
        """Test : prix négatif doit échouer."""
        with self.assertRaises(ValueError):
            self.cart.add_item("Article", -10)
    
    def test_zero_quantity_raises_error(self):
        """Test : quantité zéro doit échouer."""
        with self.assertRaises(ValueError):
            self.cart.add_item("Article", 10, quantity=0)

## Exercice 3

Pour finir, dans cette exercice, une classe `UserManager` permet de gérer des collections d'utilisateurs a travers des objets.
On peut ajouter des utilisateurs, modifier leur age, ou encore filtrer les utilisateurs selon leur age..



In [ ]:
class UserManager:
    """Gère une collection d'utilisateurs."""
    
    def __init__(self):
        self.users = {}
    
    def add_user(self, user_id, name, age):
        """Ajoute un utilisateur."""      
        self.users[user_id] = {'name': name, 'age': age}
    
    def get_user(self, user_id):
        """Récupère un utilisateur."""
        return self.users.get(user_id)
    
    def update_age(self, user_id, new_age):
        """Met à jour l'âge d'un utilisateur."""
        self.users[user_id]['age'] = new_age
    
    def get_users_by_age_range(self, min_age, max_age):
        """Retourne les utilisateurs dans une tranche d'âge."""
        return [
            user for user in self.users.values()
            if min_age <= user['age'] <= max_age
        ]

## Consignes

Créez une classe TestUserManager qui teste les scénarios suivants :
Tests d'ajout d'utilisateurs :

- Tester l'ajout d'un utilisateur valide et vérifier qu'on peut le récupérer
- Tester que count_users() retourne 0 pour un gestionnaire vide
- Tester que count_users() s'incrémente correctement après ajout
- Tester qu'ajouter un utilisateur avec un ID existant lève une ValueError
- Tester qu'ajouter un utilisateur avec un âge négatif lève une ValueError
- Tester qu'ajouter un utilisateur avec un nom vide lève une ValueError

Tests de récupération :
- Tester la récupération d'un utilisateur existant
- Tester que récupérer un utilisateur inexistant retourne None

Tests de suppression :
- Tester la suppression d'un utilisateur existant
- Tester que count_users() décrémente après suppression
- Tester que supprimer un utilisateur inexistant lève une KeyError

Tests de mise à jour :
- Tester la mise à jour de l'âge d'un utilisateur
- Tester qu'on ne peut pas mettre un âge négatif
- Tester qu'on ne peut pas mettre à jour un utilisateur inexistant

Tests de filtrage :
- Tester get_users_by_age_range() avec plusieurs utilisateurs
- Tester que le filtrage retourne une liste vide si aucun utilisateur ne correspond
- Tester les bornes (min_age et max_age sont inclus)

**Indices**: Assertions à utiliser : `assertEqual()`, `assertIsNone()`, `assertIsNotNone()`, `assertRaises()`, `assertIn()`, `assertGreater()`

Conseil avancé : Utilisez tearDown() pour nettoyer les utilisateurs après chaque test si nécessaire !

## Correction

In [ ]:
class UserManager:
    """Gère une collection d'utilisateurs."""
    
    def __init__(self):
        self.users = {}
    
    def add_user(self, user_id, name, age):
        """Ajoute un utilisateur."""
        if user_id in self.users:
            raise ValueError("Utilisateur existe déjà")
        if age < 0:
            raise ValueError("L'âge ne peut pas être négatif")
        
        self.users[user_id] = {'name': name, 'age': age}
    
    def get_user(self, user_id):
        """Récupère un utilisateur."""
        return self.users.get(user_id)
    
    def update_age(self, user_id, new_age):
        """Met à jour l'âge d'un utilisateur."""
        if user_id not in self.users:
            raise KeyError("Utilisateur non trouvé")
        if new_age < 0:
            raise ValueError("L'âge ne peut pas être négatif")
        
        self.users[user_id]['age'] = new_age
    
    def get_users_by_age_range(self, min_age, max_age):
        """Retourne les utilisateurs dans une tranche d'âge."""
        return [
            user for user in self.users.values()
            if min_age <= user['age'] <= max_age
        ]

In [ ]:
class TestUserManager(unittest.TestCase):
    """Tests pour le gestionnaire d'utilisateurs."""
    
    def setUp(self):
        self.manager = UserManager()
    
    def test_add_user(self):
        """Test : ajouter un utilisateur."""
        self.manager.add_user(1, "Alice", 25)
        user = self.manager.get_user(1)
        
        self.assertIsNotNone(user)
        self.assertEqual(user['name'], "Alice")
        self.assertEqual(user['age'], 25)
    
    def test_add_duplicate_user_raises_error(self):
        """Test : ajouter un utilisateur en double doit échouer."""
        self.manager.add_user(1, "Alice", 25)
        with self.assertRaises(ValueError):
            self.manager.add_user(1, "Bob", 30)
    
    def test_get_nonexistent_user(self):
        """Test : récupérer un utilisateur inexistant."""
        user = self.manager.get_user(999)
        self.assertIsNone(user)
    
    def test_update_age(self):
        """Test : mettre à jour l'âge."""
        self.manager.add_user(1, "Alice", 25)
        self.manager.update_age(1, 26)
        
        user = self.manager.get_user(1)
        self.assertEqual(user['age'], 26)
    
    def test_update_nonexistent_user_raises_error(self):
        """Test : mettre à jour un utilisateur inexistant."""
        with self.assertRaises(KeyError):
            self.manager.update_age(999, 30)
    
    def test_get_users_by_age_range(self):
        """Test : filtrer par tranche d'âge."""
        self.manager.add_user(1, "Alice", 25)
        self.manager.add_user(2, "Bob", 30)
        self.manager.add_user(3, "Charlie", 35)
        
        users = self.manager.get_users_by_age_range(28, 32)
        self.assertEqual(len(users), 1)
        self.assertEqual(users[0]['name'], "Bob")
    
    def tearDown(self):
        """Nettoyage après chaque test (optionnel ici)."""
        self.manager.users.clear()